<a href="https://colab.research.google.com/github/HariniRJ/bio-lab/blob/main/particle_swarm_HARINI__R_J_1BM24CS364.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random

# Multi-Crop Irrigation Scheduling using PSO

# Crop details
crops = ["Rice", "Wheat", "Maize", "Cotton"]

# Maximum water required by each crop
water_required = [500, 400, 300, 350]

# Benefit obtained per litre of water
crop_benefit = [8, 7, 6, 5]

# Total available water
available_water = 1000


# PSO parameters
NUM_PARTICLES = 20
ITERATIONS = 100
INERTIA = 0.7
COGNITIVE = 1.5
SOCIAL = 1.5


# --------------------------------
# Create a random particle
# --------------------------------

def create_particle():

    position = [
        random.uniform(0, water_required[i])
        for i in range(len(crops))
    ]

    velocity = [
        random.uniform(-50, 50)
        for _ in range(len(crops))
    ]

    return position, velocity


# --------------------------------
# Fitness function
# --------------------------------

def fitness(position):

    total_water = sum(position)

    # If water limit is exceeded
    if total_water > available_water:
        return 0

    benefit = sum(
        position[i] * crop_benefit[i]
        for i in range(len(crops))
    )

    return benefit


# --------------------------------
# Create initial particles
# --------------------------------

particles = []

for _ in range(NUM_PARTICLES):

    position, velocity = create_particle()

    particles.append({
        "position": position,
        "velocity": velocity,
        "pbest": position.copy(),
        "pbest_fitness": fitness(position)
    })


# --------------------------------
# Find initial global best
# --------------------------------

gbest = max(
    particles,
    key=lambda p: p["pbest_fitness"]
)

gbest_position = gbest["pbest"].copy()
gbest_fitness = gbest["pbest_fitness"]


# --------------------------------
# PSO Algorithm
# --------------------------------

for iteration in range(ITERATIONS):

    for particle in particles:

        position = particle["position"]
        velocity = particle["velocity"]

        # Update velocity
        for i in range(len(crops)):

            r1 = random.random()
            r2 = random.random()

            velocity[i] = (
                INERTIA * velocity[i]
                + COGNITIVE * r1 *
                (particle["pbest"][i] - position[i])
                + SOCIAL * r2 *
                (gbest_position[i] - position[i])
            )

        # Update position
        for i in range(len(crops)):

            position[i] += velocity[i]

            # Keep water allocation within crop limit
            position[i] = max(
                0,
                min(position[i], water_required[i])
            )

        # If total water exceeds available water,
        # scale the allocation down
        total_water = sum(position)

        if total_water > available_water:

            factor = available_water / total_water

            for i in range(len(crops)):
                position[i] *= factor

        # Calculate new fitness
        current_fitness = fitness(position)

        # Update personal best
        if current_fitness > particle["pbest_fitness"]:

            particle["pbest"] = position.copy()
            particle["pbest_fitness"] = current_fitness

        # Update global best
        if current_fitness > gbest_fitness:

            gbest_position = position.copy()
            gbest_fitness = current_fitness

    # Print progress
    if (iteration + 1) % 10 == 0:

        print(
            f"Iteration {iteration + 1}: "
            f"Fitness = {gbest_fitness:.2f}"
        )


# --------------------------------
# Display final solution
# --------------------------------

print("\nOptimal Irrigation Schedule")
print("--------------------------------")

total_water = 0
total_benefit = 0

for i in range(len(crops)):

    allocation = round(gbest_position[i], 2)

    benefit = allocation * crop_benefit[i]

    print(
        f"{crops[i]}: "
        f"Water = {allocation} litres, "
        f"Benefit = {benefit:.2f}"
    )

    total_water += allocation
    total_benefit += benefit


print("--------------------------------")
print("Total Water Used:", round(total_water, 2), "litres")
print("Available Water:", available_water, "litres")
print("Total Crop Benefit:", round(total_benefit, 2))

Iteration 10: Fitness = 7398.64
Iteration 20: Fitness = 7399.90
Iteration 30: Fitness = 7399.99
Iteration 40: Fitness = 7399.99
Iteration 50: Fitness = 7400.00
Iteration 60: Fitness = 7400.00
Iteration 70: Fitness = 7400.00
Iteration 80: Fitness = 7400.00
Iteration 90: Fitness = 7400.00
Iteration 100: Fitness = 7400.00

Optimal Irrigation Schedule
--------------------------------
Rice: Water = 500.0 litres, Benefit = 4000.00
Wheat: Water = 400.0 litres, Benefit = 2800.00
Maize: Water = 100.0 litres, Benefit = 600.00
Cotton: Water = 0.0 litres, Benefit = 0.00
--------------------------------
Total Water Used: 1000.0 litres
Available Water: 1000 litres
Total Crop Benefit: 7400.0
